# 09 - Engenharia de Features: Itens de Pedido + Rastreamento
**Squad 1 — Data Quality em Tempo Real | ML em Streaming**
**Responsável:** Riuler José Silvério
**Problema:** P1 · Anomalias em pedidos e itens · **Modelo A:** Isolation Forest

### Objetivo
Transformar as Silvers em uma tabela **numérica, sem nulos, com uma linha por item**, pronta para o Isolation Forest. As decisões vêm da EDA (notebook `08`).

### Saída
Delta `squad1/dupla1/ml/features_itens_pedido`, recalculada e sobrescrita a cada execução:
- **Chaves e controle:** `id_item_pedido`, `id_pedido`, `sku`, `lote`, `data_lote`, `split` (treino/teste)
- **Features do modelo:** lista `FEATURES_MODELO` abaixo
- **Avaliação, fora do treino:** `rotulo_fraco` e `qtd_regras_falhas`

### Features
| Grupo | Feature | Por quê |
|---|---|---|
| Item | `log_preco` | Preço com log com sinal; mantém os negativos distinguíveis |
| Item | `quantidade` | Pega quantidade 0 e quantidades fora do comum |
| Item | `razao_desconto` | Desconto ÷ preço. É a feature mais forte (bimodal: 0–30% ou >100%) |
| Item | `log_valor_liquido_item` | (preço − desconto) × quantidade, a mesma conta da regra 7 |
| Pedido | `n_itens_pedido`, `n_skus_pedido`, `qtd_total_pedido` | Tamanho do pedido |
| Pedido | `log_valor_liquido_pedido`, `participacao_item` | O peso do item dentro do seu pedido |
| Rastreamento | `n_eventos_rastreio`, `n_transportadoras` | Volume de eventos; a regra 10 exige transportadora única |
| Rastreamento | `horas_total_rastreio`, `max_horas_entre_eventos`, `media_horas_entre_eventos` | "Horas entre eventos", pedidas no plano |
| Rastreamento | `qtd_regressoes_status`, `teve_extraviado`, `status_mais_avancado` | Sequência fora de ordem e extravio |
| Indicadores | `tem_rastreio`, `tem_intervalo_rastreio` | Diferenciam "0 de verdade" de "sem dado" |

In [0]:
# Credenciais via Secret Scope (nunca no código)
adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

_conta = f"{adls_storage_account}.dfs.core.windows.net"
adls_options = {
    f"fs.azure.account.auth.type.{_conta}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{_conta}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{_conta}": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{_conta}": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{_conta}": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

base_squad1 = f"abfss://squad1@{_conta}"
caminhos = {
    "silver_itens": f"{base_squad1}/dupla1/silver/ecommerce_itens_pedido",
    "silver_rastreamento": f"{base_squad1}/dupla1/silver/ecommerce_rastreamento_entregas",
    "silver_produtos": f"{base_squad1}/silver/ecommerce_produtos",          # outra dupla, só leitura
    "features_itens": f"{base_squad1}/dupla1/ml/features_itens_pedido",     # saída do notebook 09
}
print("Credenciais e caminhos prontos.")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Regras por linha da Silver de itens (True = passou, False = falhou, null = não avaliada)
REGRAS_ITENS = [
    "tecnica_1_pk_valida", "tecnica_2_fk_pedido_valido", "tecnica_3_fk_sku_valido",
    "tecnica_4_quantidade_valida", "tecnica_5_preco_valido",
    "negocio_6_desconto_nao_maior_preco", "negocio_8_desconto_nao_negativo",
    "negocio_9_desconto_dentro_limite",
]
ORDEM_STATUS = {"em separacao": 1, "coletado": 2, "em transito": 3, "saiu para entrega": 4, "entregue": 5}

def com_lote(df):
    """Extrai o micro-lote (AAAA/MM/DD/HHMMSS) do caminho de origem."""
    lote = F.regexp_extract("bronze_source_file", r"(\d{4}/\d{2}/\d{2}/\d{6})", 1)
    return (df.withColumn("lote", lote)
              .withColumn("dt_lote", F.to_timestamp("lote", "yyyy/MM/dd/HHmmss"))
              .withColumn("data_lote", F.to_date("dt_lote")))

def sem_duplicatas(df, chave):
    """A Silver é append: fica a versão mais recente de cada chave."""
    w = Window.partitionBy(chave).orderBy(F.col("silver_processed_at").desc())
    return df.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")

def status_normalizado(col):
    """'EM TRANSITO', 'em_transito' e 'Em transito' viram 'em transito'."""
    return F.trim(F.lower(F.regexp_replace(col, "_", " ")))

### Etapa 1 — Leitura e limpeza mínima

- Deduplica pela chave, mantendo a versão mais recente (a Silver é append).
- Extrai o micro-lote do caminho de origem.
- Normaliza o status de rastreamento.

In [0]:
ler = lambda p: spark.read.format("delta").options(**adls_options).load(p)

itens = com_lote(sem_duplicatas(ler(caminhos["silver_itens"]), "id_item_pedido"))
rastr = (com_lote(sem_duplicatas(ler(caminhos["silver_rastreamento"]), "id_rastreamento"))
         .withColumn("status_norm", status_normalizado(F.col("status_entrega"))))

print(f"Itens: {itens.count():,} | Eventos de rastreamento: {rastr.count():,}")

### Etapa 2 — Features por item

`log_com_sinal(x) = sinal(x) · log(1 + |x|)`: comprime a cauda longa e mantém zero e negativos separados. Na `razao_desconto`, o divisor tem piso de R$ 0,01, para não dividir por zero quando o preço é 0.

In [0]:
def log_com_sinal(c):
    return F.signum(c) * F.log1p(F.abs(c))

itens_f = (itens
    .withColumn("razao_desconto", F.col("desconto_aplicado") / F.greatest(F.abs("preco_unitario"), F.lit(0.01)))
    .withColumn("valor_liquido_item", (F.col("preco_unitario") - F.col("desconto_aplicado")) * F.col("quantidade"))
    .withColumn("log_preco", log_com_sinal(F.col("preco_unitario")))
    .withColumn("log_valor_liquido_item", log_com_sinal(F.col("valor_liquido_item"))))

### Etapa 3 — Features por pedido (a partir dos itens)

Calculadas com janela por `id_pedido` e repetidas em cada item do pedido.

In [0]:
w_ped = Window.partitionBy("id_pedido")
itens_f = (itens_f
    .withColumn("n_itens_pedido", F.count("*").over(w_ped))
    .withColumn("n_skus_pedido", F.size(F.collect_set("sku").over(w_ped)))
    .withColumn("qtd_total_pedido", F.sum("quantidade").over(w_ped))
    .withColumn("valor_liquido_pedido", F.sum("valor_liquido_item").over(w_ped))
    .withColumn("log_valor_liquido_pedido", log_com_sinal(F.col("valor_liquido_pedido")))
    .withColumn("participacao_item",
                F.col("valor_liquido_item") / F.greatest(F.abs("valor_liquido_pedido"), F.lit(0.01))))

### Etapa 4 — Features de rastreamento por pedido

Os eventos são ordenados por `dt_evento` dentro do pedido, e o tempo desde o evento anterior é calculado em horas. Há **regressão** quando o status volta na sequência `em separacao → coletado → em transito → saiu para entrega → entregue`. O status `extraviado` fica fora da sequência e vira um indicador próprio.

In [0]:
mapa_ordem = F.create_map(*[F.lit(x) for kv in ORDEM_STATUS.items() for x in kv])
w_ev = Window.partitionBy("id_pedido_ecommerce").orderBy("dt_evento", "id_rastreamento")

ev = (rastr
    .withColumn("ordem_status", mapa_ordem[F.col("status_norm")])
    .withColumn("_ordem_ant", F.lag("ordem_status").over(w_ev))
    .withColumn("horas_desde_anterior",
                (F.col("dt_evento").cast("long") - F.lag("dt_evento").over(w_ev).cast("long")) / 3600.0)
    .withColumn("regressao", (F.col("ordem_status") < F.col("_ordem_ant")).cast("int")))

rastr_ped = (ev.groupBy(F.col("id_pedido_ecommerce").alias("id_pedido")).agg(
    F.count("*").alias("n_eventos_rastreio"),
    F.countDistinct("id_transportadora").alias("n_transportadoras"),
    ((F.max("dt_evento").cast("long") - F.min("dt_evento").cast("long")) / 3600.0).alias("horas_total_rastreio"),
    F.max("horas_desde_anterior").alias("max_horas_entre_eventos"),
    F.avg("horas_desde_anterior").alias("media_horas_entre_eventos"),
    F.coalesce(F.sum("regressao"), F.lit(0)).alias("qtd_regressoes_status"),
    F.max((F.col("status_norm") == "extraviado").cast("int")).alias("teve_extraviado"),
    F.coalesce(F.max("ordem_status"), F.lit(0)).alias("status_mais_avancado"),
).withColumn("tem_rastreio", F.lit(1)))

print(f"Pedidos com rastreamento: {rastr_ped.count():,}")

### Etapa 5 — Junção, nulos, rótulo fraco e divisão treino × teste

- **Nulos:** quem não tem rastreamento ou tem 1 evento só recebe 0, e os indicadores `tem_rastreio` e `tem_intervalo_rastreio` deixam isso visível para o modelo.
- **Rótulo fraco:** sai das regras por linha da Silver. Fica na tabela **só para avaliação**.
- **Split temporal:** os dias de lote que somam até cerca de 70% dos itens vão para o treino, e os dias seguintes vão para o teste. Assim, o teste simula os micro-lotes "não vistos". Como o corte é por dia inteiro, a proporção real depende do volume de cada dia: na execução de 07/11, o corte ficou em 07/10, com 8.075 itens no treino e 6.121 no teste (57% / 43%), porque o dia 08/10 sozinho tem 5.250 itens.

> **Limitação conhecida:** as features de rastreamento usam todos os eventos já gravados. No streaming real, um item chega quando o rastreio ainda está no começo. Ao pontuar um micro-lote, calcule o rastreamento só com os eventos até aquele lote. Isso fica para o notebook do modelo.

In [0]:
FEATURES_RASTREIO = ["n_eventos_rastreio", "n_transportadoras", "horas_total_rastreio", "max_horas_entre_eventos",
                     "media_horas_entre_eventos", "qtd_regressoes_status", "teve_extraviado",
                     "status_mais_avancado", "tem_rastreio"]

df = itens_f.join(rastr_ped, "id_pedido", "left").fillna(0, subset=FEATURES_RASTREIO)
df = df.withColumn("tem_intervalo_rastreio", (F.col("n_eventos_rastreio") >= 2).cast("int"))

falhou = [F.when(F.col(r) == False, 1).otherwise(0) for r in REGRAS_ITENS]   # null = não avaliada = não conta
df = (df.withColumn("qtd_regras_falhas", sum(falhou))
        .withColumn("rotulo_fraco", (F.col("qtd_regras_falhas") > 0).cast("int")))

# Split temporal por dia de lote (~70% dos itens no treino)
por_dia = df.groupBy("data_lote").count().orderBy("data_lote").toPandas()
por_dia["acum"] = por_dia["count"].cumsum() / por_dia["count"].sum()
dias_treino = por_dia.loc[por_dia.acum <= 0.70, "data_lote"].tolist() or por_dia.data_lote.iloc[:1].tolist()
corte = max(dias_treino)
df = df.withColumn("split", F.when(F.col("data_lote") <= F.lit(corte), "treino").otherwise("teste"))
print(f"Corte do treino: até {corte} | dias de treino: {len(dias_treino)} de {len(por_dia)}")

### Etapa 6 — Checagens antes de gravar

O Isolation Forest falha com nulos ou infinitos. O processo para aqui se alguma feature tiver um dos dois, ou se o número de linhas for diferente do número de itens.

In [0]:
FEATURES_MODELO = [
    "log_preco", "quantidade", "razao_desconto", "log_valor_liquido_item",
    "n_itens_pedido", "n_skus_pedido", "qtd_total_pedido", "log_valor_liquido_pedido", "participacao_item",
    "n_eventos_rastreio", "n_transportadoras", "horas_total_rastreio", "max_horas_entre_eventos",
    "media_horas_entre_eventos", "qtd_regressoes_status", "teve_extraviado", "status_mais_avancado",
    "tem_rastreio", "tem_intervalo_rastreio",
]
CHAVES = ["id_item_pedido", "id_pedido", "sku", "lote", "data_lote", "split"]
AVALIACAO = ["rotulo_fraco", "qtd_regras_falhas"]
LEGIVEIS = ["preco_unitario", "desconto_aplicado", "valor_liquido_item", "valor_liquido_pedido"]  # para interpretar alertas

df_out = df.select(*CHAVES, *FEATURES_MODELO, *AVALIACAO, *LEGIVEIS).withColumn("features_gerado_em", F.current_timestamp())

problemas = df_out.select([F.sum((F.col(c).isNull() | F.isnan(c) |
                                  F.col(c).isin(float("inf"), float("-inf"))).cast("int")).alias(c)
                           for c in FEATURES_MODELO]).first().asDict()
ruins = {k: v for k, v in problemas.items() if v}
n_out, n_itens = df_out.count(), itens.count()
assert not ruins, f"Features com nulo/NaN/inf: {ruins}"
assert n_out == n_itens, f"Linhas {n_out} ≠ itens {n_itens}"

print(f"OK: {n_out:,} linhas, {len(FEATURES_MODELO)} features, sem nulos nem infinitos.")
display(df_out.groupBy("split").agg(F.count("*").alias("itens"), F.sum("rotulo_fraco").alias("rotulados"),
                                    F.round(F.avg("rotulo_fraco") * 100, 2).alias("pct_rotulado")).orderBy("split"))
display(df_out.select(FEATURES_MODELO).summary("min", "25%", "50%", "75%", "max"))

### Etapa 7 — Gravação

Usa `overwrite`: a tabela é derivada e recalculada por inteiro a partir das Silvers, sem histórico próprio. O caminho fica dentro de `squad1/dupla1/`, sem tocar em pastas de outras duplas.

In [0]:
import json

(df_out.write.format("delta").options(**adls_options)
    .mode("overwrite").option("overwriteSchema", "true")
    .save(caminhos["features_itens"]))
print(f"[OK] Features gravadas em {caminhos['features_itens']}")

resultado = {"linhas": n_out, "features": FEATURES_MODELO, "corte_treino": str(corte)}
dbutils.notebook.exit(json.dumps(resultado))

### Próximo passo: notebook do modelo (`10`)
1. Ler `features_itens_pedido` e treinar o `IsolationForest` em `split = 'treino'`, com `contamination` perto de 0,09 (a taxa do rótulo), em três versões:
   - **A (principal):** 9 features de item e pedido;
   - **A sem valor:** a A sem `log_valor_liquido_item` e `participacao_item`;
   - **A + rastreio:** as 19 features.
2. Pontuar o teste e cada micro-lote novo (`score_samples`); o alerta é o score abaixo do limiar do treino.
3. Avaliar contra o `rotulo_fraco` com Precision, Recall, F1 e PR-AUC e calcular o % de registros alertados.
4. Listar os alertas **sem** rótulo (preços altos, quantidades 6 e 12): são o ganho do modelo sobre as regras, ou falsos positivos a explicar.
5. Comparar com o Autoencoder de Guilherme & Carlos.